In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os 
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "core"


In [ ]:
load_dotenv()

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=os.getenv("SPOTIFY_CLIENT_ID"),
        client_secret=os.getenv("SPOTIFY_CLIENT_SECRET")
    )
)

In [ ]:
results = sp.search(q="BTS", type="artist", limit=10)

artist = next(
    item for item in results["artists"]["items"]
    if item["name"].casefold() == "bts"
)

artist_id = artist["id"]
print(artist_id)


In [ ]:
artist_id = "3Nrfpe0tUJi4K4DXYWgMUX"

albums = sp.artist_albums(
    artist_id,
    album_type="album",
    limit=10
)

for album in albums["items"]:
    print(album["name"], "-", album["id"])

In [ ]:
album_id = "3ukkRHDHbN8tNRPKsGZR1h"
tracks = sp.album_tracks(album_id)

for track in tracks["items"]:
    print(track["name"])

In [ ]:
# Instead of manually copying album IDs, loop through every album automatically:

albums = sp.artist_albums(
    artist_id,
    album_type="album",
    limit=10
)

for album in albums["items"]:
    print(f"\nAlbum: {album["name"]}")

    tracks = sp.album_tracks(album["id"])

    for track in tracks["items"]:
        print(f"  - {track['name']}")

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

album_rows = []
track_rows = []

artist_id = "3Nrfpe0tUJi4K4DXYWgMUX"

albums = sp.artist_albums(
    artist_id,
    include_groups="album,single",
    limit=10
)

all_albums = albums["items"]

while albums["next"]:
    albums = sp.next(albums)
    all_albums.extend(albums["items"])

for album in all_albums:
    album_id = album["id"]

    album_rows.append({
        "album_id": album_id,
        "album_name": album["name"],
        "album_type": album["album_type"],
        "release_date": album["release_date"],
        "total_tracks": album["total_tracks"],
        "spotify_url": album["external_urls"]["spotify"],
        "image_url": album["images"][0]["url"] if album["images"] else None
    })

    tracks = sp.album_tracks(album_id)

    for track in tracks["items"]:
        track_rows.append({
            "track_id": track["id"],
            "track_name": track["name"],
            "album_id": album_id,
            "album_name": album["name"],
            "track_number": track["track_number"],
            "duration_ms": track["duration_ms"],
            "explicit": track["explicit"],
            "spotify_url": track["external_urls"]["spotify"]
        })

albums_df = pd.DataFrame(album_rows)
tracks_df = pd.DataFrame(track_rows)

albums_df.to_csv(RAW_DIR / "spotify_albums.csv", index=False)
tracks_df.to_csv(RAW_DIR / "spotify_tracks.csv", index=False)

print("Saved albums:", albums_df.shape)
print("Saved tracks:", tracks_df.shape)

In [ ]:
albums_df.head()


In [ ]:
tracks_df.head()